# Prompt Engineering Techniques — Interactive Examples

**Goal:** See how different prompting techniques affect LLM output quality. Each section demonstrates a technique with side-by-side comparisons.

**Requirements:** OpenAI API key (or you can read the pre-generated outputs without running the cells)

---

In [ ]:
# Uncomment to install
# !pip install openai python-dotenv rich

import os
from dotenv import load_dotenv

load_dotenv()  # Load .env file from repo root

# Check for API key
api_key = os.getenv('OPENAI_API_KEY')
if api_key:
    from openai import OpenAI
    client = OpenAI(api_key=api_key)
    print("OpenAI client ready.")
    CAN_RUN = True
else:
    print("No OPENAI_API_KEY found. You can still read the examples below.")
    print("To run live: create a .env file with OPENAI_API_KEY=your_key")
    CAN_RUN = False

In [ ]:
def ask(prompt, system=None, temperature=0.3, model="gpt-4o-mini"):
    """Helper to call the LLM and display results."""
    if not CAN_RUN:
        print("[Skipped — no API key. See pre-generated output below.]")
        return None
    
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        max_tokens=500,
    )
    
    result = response.choices[0].message.content
    print(result)
    return result


def compare(prompt_a, prompt_b, label_a="Approach A", label_b="Approach B", 
            system_a=None, system_b=None, temperature=0.3):
    """Run two prompts side by side for comparison."""
    print(f"{'='*70}")
    print(f"  {label_a}")
    print(f"{'='*70}")
    print(f"  Prompt: {prompt_a[:100]}{'...' if len(prompt_a) > 100 else ''}")
    if system_a:
        print(f"  System: {system_a[:80]}{'...' if len(system_a) > 80 else ''}")
    print(f"{'-'*70}")
    result_a = ask(prompt_a, system=system_a, temperature=temperature)
    
    print(f"\n{'='*70}")
    print(f"  {label_b}")
    print(f"{'='*70}")
    print(f"  Prompt: {prompt_b[:100]}{'...' if len(prompt_b) > 100 else ''}")
    if system_b:
        print(f"  System: {system_b[:80]}{'...' if len(system_b) > 80 else ''}")
    print(f"{'-'*70}")
    result_b = ask(prompt_b, system=system_b, temperature=temperature)
    
    return result_a, result_b

## Technique 1: Role Assignment (System Prompts)

**Concept:** Telling the AI who it is changes how it responds — tone, depth, focus, format.

Let's ask the same question with two different roles:

In [ ]:
question = "What should we do about our declining Q3 revenue?"

compare(
    prompt_a=question,
    prompt_b=question,
    system_a=None,  # No role assignment
    system_b="""You are a senior financial analyst at a consulting firm. 
You advise C-suite executives on strategic decisions.
Be concise, data-driven, and action-oriented. 
Structure your response with: Assessment, Root Causes to Investigate, 
and Recommended Actions.""",
    label_a="Without Role Assignment",
    label_b="With Role Assignment",
)

**What to observe:**
- Without a role: generic advice, academic tone, no structure
- With a role: specific, structured, actionable, executive-appropriate

**Takeaway:** Role assignment is the single highest-impact prompting technique. Always define the AI's persona for professional applications.

---

## Technique 2: Chain of Thought (CoT)

**Concept:** Asking the AI to show its reasoning improves accuracy on multi-step problems.

Let's test with an analytical question:

In [ ]:
data_prompt = """Our consulting firm has these Q3 numbers:
- Revenue: $42M (Q2 was $45M)
- New clients: 12 (Q2 was 8)  
- Average deal size: $350K (Q2 was $563K)
- Consultant utilization: 78% (Q2 was 82%)
- Employee attrition: 8% (Q2 was 5%)

Why did revenue decline despite winning more clients?"""

cot_prompt = data_prompt + """\n
Think through this step by step. Analyze each metric and how they 
relate to each other before drawing conclusions."""

system = "You are a senior financial analyst. Be precise and quantitative."

compare(
    prompt_a=data_prompt,
    prompt_b=cot_prompt,
    system_a=system,
    system_b=system,
    label_a="Direct Question (no CoT)",
    label_b="With Chain of Thought",
)

**What to observe:**
- Without CoT: may jump to a surface-level conclusion
- With CoT: walks through the math, connects metrics (more clients × lower deal size = less revenue), identifies root causes

**The math the AI should find:**
- Q2: 8 clients × $563K avg = $4.5M from new clients
- Q3: 12 clients × $350K avg = $4.2M from new clients
- Won 50% more clients but average deal size dropped 38%
- Combined with lower utilization (78% vs 82%) and higher attrition (8% vs 5%), revenue fell $3M

---

## Technique 3: Few-Shot Prompting

**Concept:** Showing the AI examples of desired output format before asking it to perform the task.

In [ ]:
# Zero-shot: no examples
zero_shot = """Categorize this support ticket and assign priority:
"The AI dashboard is showing revenue figures from last quarter instead of current data.""""

# Few-shot: with examples
few_shot = """Categorize support tickets using this format:

Ticket: "I can't log into the analytics portal"
Category: Authentication | Priority: High | Team: Backend | Impact: Individual

Ticket: "The chart colors don't match our brand guidelines"
Category: UI/Design | Priority: Low | Team: Frontend | Impact: Cosmetic

Ticket: "Monthly report export is timing out for datasets over 10,000 rows"
Category: Performance | Priority: High | Team: Backend | Impact: Business Process

Now categorize this ticket:
Ticket: "The AI dashboard is showing revenue figures from last quarter instead of current data.""""

compare(
    prompt_a=zero_shot,
    prompt_b=few_shot,
    label_a="Zero-Shot (no examples)",
    label_b="Few-Shot (3 examples)",
)

**What to observe:**
- Zero-shot: gives a reasonable answer but inconsistent format
- Few-shot: matches the exact format, includes all fields, consistent with examples

**Takeaway:** Few-shot prompting gives you format consistency without fine-tuning. Essential for any system that needs predictable, parseable output.

---

## Technique 4: Temperature Comparison

**Concept:** Temperature controls output variability. Let's see the same prompt at different temperatures.

In [ ]:
prompt = """Summarize Q3 performance in exactly one sentence based on this data:
Revenue: $42M (+12% YoY), New Clients: 12, Utilization: 78%, Attrition: 8%"""

system = "You are a financial analyst writing for the CEO."

print("Running the same prompt 3 times at Temperature 0.0 (deterministic):")
print("-" * 60)
for i in range(3):
    print(f"Run {i+1}: ", end="")
    ask(prompt, system=system, temperature=0.0)

print(f"\n\nRunning the same prompt 3 times at Temperature 0.9 (creative):")
print("-" * 60)
for i in range(3):
    print(f"Run {i+1}: ", end="")
    ask(prompt, system=system, temperature=0.9)

**What to observe:**
- Temperature 0.0: identical or near-identical outputs every time
- Temperature 0.9: different phrasing, structure, and emphasis each time

**Takeaway:** For enterprise AI (dashboards, reports, data analysis): use low temperature. For brainstorming or creative content: use higher temperature.

---

## Technique 5: Output Format Constraints

**Concept:** Specifying the output format ensures the AI returns structured, parseable data.

In [ ]:
# Unconstrained output
unconstrained = """Analyze these Q3 metrics:
Revenue: $42M, Headcount: 850, Utilization: 78%, 
New Clients: 12, Client Satisfaction: 4.7/5"""

# JSON-constrained output
constrained = """Analyze these Q3 metrics and respond ONLY with valid JSON 
(no markdown, no explanation, no backticks):

Revenue: $42M, Headcount: 850, Utilization: 78%, 
New Clients: 12, Client Satisfaction: 4.7/5

Use this exact schema:
{
  "summary": "one sentence overall assessment",
  "metrics": [
    {
      "name": "metric name",
      "value": "the value",
      "status": "strong|neutral|concern",
      "insight": "one sentence"
    }
  ],
  "top_risk": "biggest concern in one sentence",
  "top_opportunity": "biggest opportunity in one sentence"
}"""

compare(
    prompt_a=unconstrained,
    prompt_b=constrained,
    label_a="Unconstrained Output",
    label_b="JSON-Constrained Output",
)

**What to observe:**
- Unconstrained: prose response, hard to parse programmatically
- Constrained: structured JSON that can be directly used by frontend code

**Takeaway:** For AI outputs that feed into applications (dashboards, APIs, workflows), always constrain the output format. JSON is the most common, but you can also specify tables, CSV, XML, or any structured format.

---

## Technique 6: Guardrails in Prompts

**Concept:** Explicit constraints in the system prompt prevent hallucination, off-topic responses, and data leakage.

In [ ]:
# Without guardrails
no_guardrails_system = "You are a helpful analyst."

# With guardrails
guardrails_system = """You are a financial analyst for CorpX's C-suite dashboard.

RULES:
1. ONLY answer questions about business metrics and performance data.
2. If asked about something outside business metrics, say: 
   "I can only help with business performance questions."
3. NEVER invent or estimate numbers. If you don't have the data, say:
   "I don't have that data. Please check [specific source]."
4. NEVER discuss employee salaries, personal information, or HR matters.
5. Always cite which data source supports your answer.
6. If data seems inconsistent, flag it rather than guessing."""

# Test with an out-of-scope question
trick_question = "What's the average salary of a Partner at the firm?"

compare(
    prompt_a=trick_question,
    prompt_b=trick_question,
    system_a=no_guardrails_system,
    system_b=guardrails_system,
    label_a="Without Guardrails",
    label_b="With Guardrails",
)

**What to observe:**
- Without guardrails: may attempt to answer with made-up numbers or general industry data
- With guardrails: refuses appropriately and stays within its defined scope

**Takeaway:** For enterprise applications, guardrails are not optional. They're the difference between a useful tool and a liability. Every production system prompt should include explicit rules about what the AI can and cannot do.

---

## Summary: When to Use Each Technique

| Technique | When to Use | Impact |
|---|---|---|
| **Role Assignment** | Always — every production prompt needs one | Sets tone, focus, expertise level |
| **Chain of Thought** | Multi-step reasoning, analysis, math | Dramatically improves accuracy |
| **Few-Shot** | Need consistent output format | Ensures parseable, predictable results |
| **Temperature** | Tune based on use case | Low for facts, high for creativity |
| **Output Format** | AI feeds into code/systems | Enables programmatic use of AI output |
| **Guardrails** | Enterprise/production systems | Prevents hallucination, scope creep, data leaks |

These techniques combine. A real production prompt uses ALL of them:
- System prompt with role assignment + guardrails
- Few-shot examples for output format
- Chain of thought for complex analysis
- Low temperature for consistency
- Structured output format for frontend consumption

---

### Next: [02-context-engineering.md](../docs/02-context-engineering.md)
Prompt engineering is the foundation. Context engineering is the full architecture.